# Pilot: internal knowledge as arbiter in knowledge conflict

Thin cells calling into the `pilot` package. Logic lives in `.py` files under
version control; this notebook is for plots and narration.

**Run the cells in order.** Each stage is idempotent and resumable — after a dead
session, re-run from the top and completed work is skipped.

Kill criteria are fixed in `pilot/config.py`. If one fires, **stop, write it up in
`RESULTS.md`, and report.** Do not adjust a threshold, do not search for a variant
that passes, do not run the next test.

## 0. Environment

In [ ]:
# Drive first: Colab sessions die, Drive survives them. All artefacts land there.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('colab:', IN_COLAB)

In [ ]:
%cd /content/drive/MyDrive
REPO = 'https://github.com/YOUR-ORG/knowledge-conflict'   # <- set this
!git clone $REPO knowledge-conflict 2>/dev/null || (cd knowledge-conflict && git pull)
%cd knowledge-conflict
!pip install -q -r requirements.txt

In [ ]:
# HF token from Colab Secrets (sidebar, key icon), named HF_TOKEN.
# Llama-3 is a gated repo. Never paste a token into a cell.
from pilot.model import hf_token
assert hf_token(), 'add HF_TOKEN to Colab Secrets and re-run'
print('token found')

In [ ]:
# Check the plumbing before spending the session on it.
!pytest -q
!pytest tests/test_torch_paths.py --run-slow -q

In [ ]:
from pilot import config
from pilot.cli import status
print(config.PATHS.root)
status()

## Test 0 — Does the resistance state exist in useful volume?

Build the fact set, then label each fact's conflict state *for this model* by asking
it closed-book 8 times at temperature 0.7.

Diagnostic — no kill criterion. But watch two numbers: the **ambiguous fraction**
(if over half the set lands between 0/8 and 6/8, the thresholds are wrong for this
model, which is a finding to raise, not a knob to turn) and the **popularity
distribution per state** (separated distributions mean log-popularity alone
separates Test 2's binary).

In [ ]:
from pilot.stages import stage00_factset
stage00_factset.run()

In [ ]:
from pilot.stages import stage01_prior
summary = stage01_prior.run()
summary

In [ ]:
from pilot import figures, io_utils
from pilot.factset import pop_bin

s = io_utils.read_json(config.PATHS.test_dir('test0') / 'summary.json')
figures.state_counts(s, config.PATHS.figures / 'test0_states.png')

facts = io_utils.read_jsonl(config.PATHS.factset)
by_id = {f['fact_id']: f for f in facts}
cases = io_utils.read_jsonl(config.PATHS.states)
log_pop = {}
for c in cases:
    f = by_id.get(c['fact_id'])
    if f and f.get('log_s_pop') is not None:
        log_pop.setdefault(c['state'], []).append(f['log_s_pop'])
figures.popularity_by_state(log_pop, config.PATHS.figures / 'test0_popularity.png')

In [ ]:
from IPython.display import Image, display
display(Image(str(config.PATHS.figures / 'test0_states.png')))
display(Image(str(config.PATHS.figures / 'test0_popularity.png')))

### Cross-check our prior labels against TriState-Bench

The vendored repo ships states screened against this exact model by a different
procedure (GAPS). Where the two disagree, at least one is wrong — the closest thing
to an external check on our 6/8 and 0/8 thresholds available without new annotation.

In [ ]:
from pilot.factset import load_tristate
from pilot.prior import agreement_with_tristate
from pilot.vendor import ensure_repo

ts = load_tristate(ensure_repo(config.PATHS.vendor))
print({k: sum(1 for f in ts if f['state'] == k)
       for k in ('correction', 'resistance', 'agreement')})
agreement_with_tristate(io_utils.read_jsonl(config.PATHS.prior), ts)

## Capture — the one expensive pass

Everything Tests 1-4 need is logged here, once: per-layer internal scores, external
scores, all output-distribution signals, top-k of both distributions, the residual
stream at every layer.

The **logit lens self-check runs first** and raises if the lens cannot reproduce the
model's own final-layer logits. In HuggingFace Llama the last hidden state has
already been through the final norm, and applying it again produces a plausible but
wrong distribution — so this is checked, not assumed. Nothing is captured if it
fails.

In [ ]:
from pilot.stages import stage02_capture
cap = stage02_capture.run()
cap['lens_check']

## Test 1 — Do internal and external answers diverge?

Layer chosen on the `layer` split; divergence reported on `train`. The report split
stays locked.

**Kill criterion:** divergence below ~10% of facts, **or** internal not beating
external on the subset where they diverge.

In [ ]:
from pilot.stages import stage03_test1
t1 = stage03_test1.run()
t1['kill']

In [ ]:
display(Image(str(config.PATHS.figures / 'test1_knowledge_by_layer.png')))
display(Image(str(config.PATHS.figures / 'test1_scatter.png')))

## Test 2 — Does the internal signal predict the *decision*?

**This is the real gate.** On correction and resistance cases only: should we
resist? Three internal predictors against seven external ones, on the same
questions with shared bootstrap resamples.

**Read `log_popularity` first.** If it matches the best internal signal, the
internal signal is a frequency detector.

**Kill criterion:** best internal does not beat best external by +0.05 AUC with
non-overlapping CIs. If it fails *narrowly*, check the error correlation — low
correlation means complementary rather than superior, and the project becomes an
ensemble method. Weaker, still viable.

In [ ]:
from pilot.stages import stage04_test2
t2 = stage04_test2.run()
t2['kill']

In [ ]:
import pandas as pd
data = io_utils.read_json(config.PATHS.test_dir('test2') / 'test2.json')
from pilot.signals import INTERNAL_SIGNALS
df = pd.DataFrame(data['auc']).T
df['kind'] = ['internal' if n in INTERNAL_SIGNALS else 'external' for n in df.index]
df.sort_values('auc', ascending=False)[['kind', 'auc', 'lo', 'hi', 'n']]

In [ ]:
display(Image(str(config.PATHS.figures / 'test2_auc.png')))
display(Image(str(config.PATHS.figures / 'test2_error_correlation.png')))

## Test 3a — Analytic reachability

τ*(a,b) = −ℓ_pri / (ℓ_ctx − ℓ_pri) is where the model's preference between a and b
flips. Theory predicts τ* ∈ (0,1) for resistance cases — interpolation suffices, and
the extrapolation regime every existing method lives in is pointed the wrong way.
Verify empirically.

In [ ]:
from pilot.stages import stage05_test3a
t3a = stage05_test3a.run()
display(Image(str(config.PATHS.figures / 'test3a_tau_star.png')))
t3a['competitor_is_ctx_top']

## Test 3b — Oracle ceiling

Sweep the power family, run the vendored baselines, then route τ per question from
the **ground-truth** state — a three-way oracle, not a one-sided gate (a one-sided
oracle measures the ceiling of the wrong method).

**Kill criterion:** oracle routing barely beats the strongest baseline. No signal
quality can exceed the oracle, so a low ceiling means there is nothing to chase.

Longest stage in the pilot. `--grid coarse` (the default, 11 τ points) is enough for
the trade-off curves; it is resumable at generation granularity.

In [ ]:
from pilot.stages import stage06_test3b
t3b = stage06_test3b.run(grid='coarse')
t3b['kill']

In [ ]:
d = io_utils.read_json(config.PATHS.test_dir('test3b') / 'test3b.json')
rows = dict(d['baselines'])
rows['oracle3'] = d['oracle']
cols = ['overall', 'correction', 'resistance', 'agreement']
pd.DataFrame(rows).T[cols].sort_values('overall', ascending=False)

In [ ]:
display(Image(str(config.PATHS.figures / 'test3b_tau_sweep.png')))
display(Image(str(config.PATHS.figures / 'test3b_tradeoff.png')))

## Test 4 — Permutation control

Shuffle the surviving signal across questions, marginal held fixed, and re-run the
routing. Gains are measured against the **best fixed τ** — against τ=1 a pure global
rescale would look like a win and pass a control designed to catch exactly that.

**Kill criterion:** the shuffled curve matches the real one. Then the signal is a
global correction-strength knob and the per-question adaptivity — the entire claim —
is doing nothing.

**Run this every time a promising result appears, not just once.**

In [ ]:
from pilot.stages import stage07_test4
t4 = stage07_test4.run(n_shuffles=5)
t4['kill']

In [ ]:
# Stricter variant: shuffle within each conflict state. Preserves any between-state
# difference in the signal's range and asks whether the within-state ordering carries
# anything.
stage07_test4.run(n_shuffles=5, within_state=True)['kill']

In [ ]:
display(Image(str(config.PATHS.figures / 'test4_permutation.png')))

## Two-pass overhead

Is the literature's "2× cost" framing real? Nearly free to answer while the model is
already loaded.

In [ ]:
from pilot.stages import stage08_timing
stage08_timing.run()

## Write-up

Fill in `RESULTS.md` **as you go**, and `DECISIONS.md` for anything decided along
the way that the spec did not determine.

Only after Tests 0-4 are written up: unlock the report split and re-run the frozen
pipeline once on it. Nothing may be tuned after that.

```bash
python -m pilot.cli unlock-report --reason 'tests 0-4 complete and written up'
python -m pilot.cli test1 --report-splits report
python -m pilot.cli test2 --report-splits report
```

In [ ]:
status()